# 🌏 Tourism Customer Analysis
## Business-Driven Data Analysis for a Travel Company

---

### 🎯 Business Problem
A travel company currently converts only **18% of customers** into package buyers despite having rich customer data. Customers were contacted **randomly** — without data-driven targeting.

The company is launching a new **Wellness Tourism Package** and wants to:
1. Identify **which customer segments** are most likely to buy
2. Understand **what drives package purchase decisions**
3. Build **customer profiles** for each of the 5 existing packages (Basic, Standard, Deluxe, Super Deluxe, King)
4. Generate **actionable business recommendations** to improve conversion rate

### 💼 Business Impact
If we can increase the conversion rate from 18% to just 25% by targeting the right customers, that represents a **38% improvement in revenue** from the same customer base — without acquiring new customers.

---

### 📋 Approach
1. Data Cleaning & Quality Assurance
2. Exploratory Data Analysis (Univariate → Bivariate → Multivariate)
3. Customer Segmentation & Profiling
4. Conversion Rate Analysis by Segment
5. Wellness Package Target Audience Identification
6. Business Recommendations

---
## 📦 Step 1: Import Libraries & Load Data

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
%matplotlib inline

# Styling
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

import warnings
warnings.filterwarnings('ignore')

# Load dataset
data = pd.read_excel('Tourism.xlsx', sheet_name=1)
df = data.copy()

print(f'Dataset Shape: {df.shape}')
print(f'Total Customers: {df.shape[0]:,}')
print(f'Total Features: {df.shape[1]}')

In [ ]:
df.head()

---
## 🔍 Step 2: Data Quality Assessment

Before any analysis, we need to understand data quality — missing values, duplicates, and data type issues.

In [ ]:
# ── 2.1 Data types and null counts ──
df.info()

In [ ]:
# ── 2.2 Statistical summary ──
df.describe(include='all').T

In [ ]:
# ── 2.3 Missing Value Heatmap — Business Priority View ──
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#e74c3c' if x > 4 else '#f39c12' if x > 1 else '#2ecc71' for x in missing_df['Missing %']]
axes[0].barh(missing_df.index, missing_df['Missing %'], color=colors)
axes[0].set_xlabel('Missing %')
axes[0].set_title('Missing Values by Column', fontweight='bold')
for i, (val, col) in enumerate(zip(missing_df['Missing %'], missing_df.index)):
    axes[0].text(val + 0.1, i, f'{val:.1f}%', va='center')

# Heatmap
sns.heatmap(df[missing_df.index].isnull(), cbar=False, cmap='Reds', ax=axes[1], yticklabels=False)
axes[1].set_title('Missing Value Pattern', fontweight='bold')

plt.tight_layout()
plt.show()
print(missing_df)

---
## 🧹 Step 3: Data Cleaning

### 3.1 — Drop CustomerID & Check Duplicates

In [ ]:
# Drop non-predictive ID column
df.drop(['CustomerID'], axis=1, inplace=True)

# Check duplicates AFTER dropping ID (IDs mask duplicate rows)
dupes = df.duplicated().sum()
print(f'Duplicate rows found (after dropping CustomerID): {dupes}')
# These are coincidental matches — no name/address to confirm real duplicates. Keeping them.

### 3.2 — Fix Data Entry Errors in Categorical Columns

In [ ]:
# 'Fe Male' is a data entry error → standardize to 'Female'
print("Before fix:", df['Gender'].value_counts().to_dict())
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
print("After fix: ", df['Gender'].value_counts().to_dict())

### 3.3 — Drop Freelancers (only 2 rows — negligible)

In [ ]:
print(f"Freelancer rows: {(df['Occupation']=='Free Lancer').sum()}")
df = df[df['Occupation'] != 'Free Lancer'].reset_index(drop=True)
print(f"Dataset size after dropping freelancers: {df.shape}")

### 3.4 — Impute Missing Values (Strategically)

In [ ]:
# ── MonthlyIncome: impute median grouped by Occupation + Designation ──
# (Income is strongly tied to job level — grouping preserves real-world logic)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(
    df.groupby(['Occupation', 'Designation'])['MonthlyIncome'].transform('median')
)

# ── Age: impute median grouped by Gender + Designation ──
df['Age'] = df['Age'].fillna(
    df.groupby(['Gender', 'Designation'])['Age'].transform('median')
)

# ── NumberOfChildrenVisiting: impute mode grouped by NumberOfPersonVisiting ──
# (children count is logically bounded by group size)
df['NumberOfChildrenVisiting'] = df.groupby('NumberOfPersonVisiting')['NumberOfChildrenVisiting']\
    .apply(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else x.median()))

# ── NumberOfFollowups: all groups have median = 4.0 → fill directly ──
df['NumberOfFollowups'] = df['NumberOfFollowups'].fillna(4.0)

# ── PreferredPropertyStar: fill with 3.0 (most common across all groups) ──
df['PreferredPropertyStar'] = df['PreferredPropertyStar'].fillna(3.0)

# ── TypeofContact: fill with mode (Self Enquiry) ──
df['TypeofContact'] = df['TypeofContact'].fillna('Self Enquiry')

# ── NumberOfTrips: impute median grouped by children + followups (correlated) ──
df['NumberOfTrips'] = df['NumberOfTrips'].fillna(
    df.groupby(['NumberOfChildrenVisiting', 'NumberOfFollowups'])['NumberOfTrips'].transform('median')
)

# ── DurationOfPitch: impute median grouped by ProductPitched + NumberOfFollowups ──
df['DurationOfPitch'] = df['DurationOfPitch'].fillna(
    df.groupby(['ProductPitched', 'NumberOfFollowups'])['DurationOfPitch'].transform('median')
)

# ── Verify: no missing values remaining ──
remaining = df.isnull().sum().sum()
print(f'✅ Missing values remaining after imputation: {remaining}')

### 3.5 — Outlier Detection & Treatment

In [ ]:
def treat_outliers(df, col, method='cap'):
    """Cap outliers at IQR whiskers instead of dropping data."""
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_before = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'{col}: {outliers_before} outliers capped → [{lower:.1f}, {upper:.1f}]')
    return df

# Treat outliers in continuous numerical columns
for col in ['DurationOfPitch', 'NumberOfTrips', 'MonthlyIncome']:
    df = treat_outliers(df, col)

# NumberOfPersonVisiting: cap at 4 (5 is essentially same trip size)
df['NumberOfPersonVisiting'] = df['NumberOfPersonVisiting'].clip(upper=4)
print(f'NumberOfPersonVisiting: capped max at 4')

In [ ]:
# ── Visualize outlier treatment results ──
num_cols = ['Age', 'DurationOfPitch', 'MonthlyIncome', 'NumberOfTrips',
            'NumberOfPersonVisiting', 'NumberOfFollowups', 'NumberOfChildrenVisiting']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), vert=False, patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.7))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Value')
axes[-1].set_visible(False)
fig.suptitle('Distribution of Numerical Features After Outlier Treatment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📊 Step 4: Exploratory Data Analysis

### 4.1 — Target Variable: Who is Actually Buying?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
counts = df['ProdTaken'].value_counts()
labels = ['Did NOT Buy\n(82%)', 'Bought Package\n(18%)']
colors = ['#e74c3c', '#2ecc71']
axes[0].pie(counts, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Overall Package Conversion Rate', fontweight='bold', fontsize=13)

# Conversion rate by product pitched
conv_by_product = df.groupby('ProductPitched')['ProdTaken'].mean().sort_values(ascending=False) * 100
bars = axes[1].bar(conv_by_product.index, conv_by_product.values,
                   color=['#2ecc71' if v == conv_by_product.max() else '#3498db' for v in conv_by_product.values],
                   edgecolor='white')
axes[1].set_title('Conversion Rate by Product Pitched', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].axhline(y=18, color='red', linestyle='--', alpha=0.7, label='Overall avg (18%)')
axes[1].legend()
for bar, val in zip(bars, conv_by_product.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold')

plt.suptitle('💰 Business Opportunity: Package Purchase Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📌 KEY INSIGHT: Basic package has the highest conversion — lowest price barrier.')
print('📌 King package has very low conversion despite premium pitch — wrong audience being targeted.')

### 4.2 — Demographic Analysis

In [ ]:
cat_cols = ['Occupation', 'Gender', 'MaritalStatus', 'CityTier', 'Designation']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    conv = df.groupby(col)['ProdTaken'].mean().sort_values(ascending=False) * 100
    bars = axes[i].bar(conv.index.astype(str), conv.values,
                       color=sns.color_palette('Set2', len(conv)), edgecolor='white')
    axes[i].axhline(y=18, color='red', linestyle='--', alpha=0.6, label='Avg 18%')
    axes[i].set_title(f'Conversion Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Conversion Rate (%)')
    axes[i].legend(fontsize=9)
    axes[i].tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, conv.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     f'{val:.0f}%', ha='center', fontsize=9, fontweight='bold')

axes[-1].set_visible(False)
fig.suptitle('🎯 Conversion Rate by Customer Demographics', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 — Income & Age: The Two Biggest Predictors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Income distribution by purchase
for label, grp in df.groupby('ProdTaken'):
    axes[0].hist(grp['MonthlyIncome'], bins=30, alpha=0.6,
                 label='Bought' if label == 1 else 'Did Not Buy',
                 color='#2ecc71' if label == 1 else '#e74c3c')
axes[0].set_title('Monthly Income Distribution by Purchase', fontweight='bold')
axes[0].set_xlabel('Monthly Income (₹)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Age distribution by purchase
for label, grp in df.groupby('ProdTaken'):
    axes[1].hist(grp['Age'], bins=20, alpha=0.6,
                 label='Bought' if label == 1 else 'Did Not Buy',
                 color='#3498db' if label == 1 else '#e74c3c')
axes[1].set_title('Age Distribution by Purchase', fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Income & Age vs. Package Purchase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print summary stats
print('💰 INCOME — Average by Purchase:')
print(df.groupby('ProdTaken')['MonthlyIncome'].mean().rename({0:'Did Not Buy', 1:'Bought'}).apply(lambda x: f'₹{x:,.0f}'))
print('\n👤 AGE — Average by Purchase:')
print(df.groupby('ProdTaken')['Age'].mean().rename({0:'Did Not Buy', 1:'Bought'}).apply(lambda x: f'{x:.1f} years'))

### 4.4 — Sales Process Analysis: Pitches & Follow-ups

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Followups vs Conversion
conv_followup = df.groupby('NumberOfFollowups')['ProdTaken'].mean() * 100
axes[0].plot(conv_followup.index, conv_followup.values, 'o-', color='#3498db', linewidth=2, markersize=8)
axes[0].fill_between(conv_followup.index, conv_followup.values, alpha=0.15, color='#3498db')
axes[0].set_title('Conversion Rate vs. # of Followups', fontweight='bold')
axes[0].set_xlabel('Number of Follow-ups')
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].axhline(y=18, color='red', linestyle='--', alpha=0.5, label='Avg 18%')
axes[0].legend()

# Duration of pitch vs conversion (binned)
df['PitchDuration_Bin'] = pd.cut(df['DurationOfPitch'], bins=[0, 10, 15, 20, 40],
                                  labels=['<10 min', '10-15 min', '15-20 min', '>20 min'])
conv_pitch = df.groupby('PitchDuration_Bin', observed=True)['ProdTaken'].mean() * 100
bars = axes[1].bar(conv_pitch.index.astype(str), conv_pitch.values,
                   color=sns.color_palette('Blues', len(conv_pitch)), edgecolor='white')
axes[1].set_title('Conversion Rate by Pitch Duration', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
for bar, val in zip(bars, conv_pitch.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', fontweight='bold')

# Pitch satisfaction vs conversion
conv_sat = df.groupby('PitchSatisfactionScore')['ProdTaken'].mean() * 100
axes[2].bar(conv_sat.index, conv_sat.values, color=sns.color_palette('RdYlGn', len(conv_sat)), edgecolor='white')
axes[2].set_title('Conversion Rate by Pitch Satisfaction Score', fontweight='bold')
axes[2].set_xlabel('Pitch Satisfaction (1=Low, 5=High)')
axes[2].set_ylabel('Conversion Rate (%)')

fig.suptitle('🤝 Sales Process Effectiveness Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5 — Correlation Heatmap

In [ ]:
num_df = df.select_dtypes(include=np.number)
corr = num_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(13, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Feature Correlation Matrix\n(focus on ProdTaken row for purchase drivers)', 
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Top correlations with target
print('\n📌 Top correlations with Package Purchase (ProdTaken):')
target_corr = corr['ProdTaken'].drop('ProdTaken').abs().sort_values(ascending=False)
print(target_corr.head(8).apply(lambda x: f'{x:.3f}'))

---
## 👥 Step 5: Customer Profiling — One Profile per Package

> **Business Goal**: For each package, identify the exact customer profile that converts — so the sales team knows *who to call* and *what to pitch*.

In [ ]:
# Build a comprehensive profile for each package
packages = ['Basic', 'Standard', 'Deluxe', 'Super Deluxe', 'King']
package_colors = {'Basic': '#3498db', 'Standard': '#2ecc71', 'Deluxe': '#9b59b6',
                  'Super Deluxe': '#e67e22', 'King': '#e74c3c'}

for pkg in packages:
    subset = df[df['ProductPitched'] == pkg]
    buyers = subset[subset['ProdTaken'] == 1]
    conv_rate = buyers.shape[0] / subset.shape[0] * 100
    
    print(f'\n{"="*60}')
    print(f'  📦 {pkg.upper()} PACKAGE CUSTOMER PROFILE')
    print(f'  Conversion Rate: {conv_rate:.1f}%  |  Total Pitched: {subset.shape[0]:,}  |  Buyers: {buyers.shape[0]}')
    print(f'{"="*60}')
    
    print(f'  💰 Income:         ₹{buyers["MonthlyIncome"].median():,.0f} median  (range: ₹{buyers["MonthlyIncome"].quantile(.25):,.0f}–₹{buyers["MonthlyIncome"].quantile(.75):,.0f})')
    print(f'  👤 Age:            {buyers["Age"].median():.0f} years median  (range: {buyers["Age"].quantile(.25):.0f}–{buyers["Age"].quantile(.75):.0f})')
    print(f'  💍 Marital Status: {buyers["MaritalStatus"].mode()[0]} ({buyers["MaritalStatus"].value_counts(normalize=True).iloc[0]*100:.0f}%)')
    print(f'  🏙️  City Tier:      Tier {buyers["CityTier"].mode()[0]} most common')
    print(f'  💼 Occupation:     {buyers["Occupation"].mode()[0]}')
    print(f'  🏅 Designation:    {buyers["Designation"].mode()[0]}')
    print(f'  👨‍👩‍👧 Group Size:    {buyers["NumberOfPersonVisiting"].mode()[0]} persons, {buyers["NumberOfChildrenVisiting"].mode()[0]:.0f} children')
    print(f'  ✈️  Avg Trips/Year: {buyers["NumberOfTrips"].median():.0f}')
    print(f'  🌍 Has Passport:   {(buyers["Passport"]==1).mean()*100:.0f}% of buyers')
    print(f'  📞 Follow-ups:     {buyers["NumberOfFollowups"].median():.0f} follow-ups median')

In [ ]:
# ── Visual: Package profiles side by side ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics = ['MonthlyIncome', 'Age', 'NumberOfTrips', 'NumberOfPersonVisiting', 'NumberOfFollowups']
metric_labels = ['Monthly Income (₹)', 'Age (years)', 'Trips/Year', 'Group Size', 'Follow-ups']

for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
    vals = [df[(df['ProductPitched']==pkg) & (df['ProdTaken']==1)][metric].median() for pkg in packages]
    colors_list = [package_colors[p] for p in packages]
    bars = axes[i].bar(packages, vals, color=colors_list, edgecolor='white', linewidth=1.5)
    axes[i].set_title(f'{label} — Buyers Only', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                     f'{val:,.0f}', ha='center', fontsize=10, fontweight='bold')

axes[-1].set_visible(False)
fig.suptitle('📦 Customer Profile Comparison Across Packages (Buyers Only)', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🧘 Step 6: Wellness Tourism Package — Target Audience Identification

This is the new product. We use patterns from existing buyers to identify the ideal target segment.

In [ ]:
# Wellness travelers tend to:
# - Be middle-aged (35–55) — health-conscious working professionals
# - Have higher income — wellness is a premium experience
# - Be passport holders — comfortable with travel
# - Travel with smaller groups or solo

# Define Wellness Target Segment
wellness_target = df[
    (df['Age'] >= 35) & (df['Age'] <= 55) &
    (df['MonthlyIncome'] >= df['MonthlyIncome'].quantile(0.5)) &  # top 50% income
    (df['Occupation'].isin(['Salaried', 'Large Business']))
].copy()

print(f'Total dataset:              {len(df):,} customers')
print(f'Wellness target segment:    {len(wellness_target):,} customers ({len(wellness_target)/len(df)*100:.1f}%)')
print(f'Baseline conversion rate:   {df["ProdTaken"].mean()*100:.1f}%')
print(f'Target segment conv. rate:  {wellness_target["ProdTaken"].mean()*100:.1f}%')

In [ ]:
# Visualize Wellness Target Segment
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution of target segment
axes[0].hist(wellness_target['Age'], bins=20, color='#27ae60', alpha=0.8, edgecolor='white')
axes[0].axvline(wellness_target['Age'].median(), color='red', linestyle='--',
                label=f'Median: {wellness_target["Age"].median():.0f} yrs')
axes[0].set_title('Age Distribution — Wellness Segment', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].legend()

# Income
axes[1].hist(wellness_target['MonthlyIncome'], bins=20, color='#8e44ad', alpha=0.8, edgecolor='white')
axes[1].axvline(wellness_target['MonthlyIncome'].median(), color='red', linestyle='--',
                label=f'Median: ₹{wellness_target["MonthlyIncome"].median():,.0f}')
axes[1].set_title('Income Distribution — Wellness Segment', fontweight='bold')
axes[1].set_xlabel('Monthly Income (₹)')
axes[1].legend()

# Conversion by city tier within segment
conv_city = wellness_target.groupby('CityTier')['ProdTaken'].mean() * 100
bars = axes[2].bar(conv_city.index.astype(str), conv_city.values,
                   color=sns.color_palette('Greens_r', len(conv_city)), edgecolor='white')
axes[2].set_title('Conversion Rate by City Tier\n(Wellness Segment)', fontweight='bold')
axes[2].set_xlabel('City Tier')
axes[2].set_ylabel('Conversion Rate (%)')
for bar, val in zip(bars, conv_city.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold')

fig.suptitle('🧘 Wellness Tourism — Target Audience Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📈 Step 7: Segment Conversion Rate Dashboard

In [ ]:
# Build a conversion rate matrix — income band × age group
df['Income_Band'] = pd.qcut(df['MonthlyIncome'], q=4,
                             labels=['Low (<20K)', 'Mid-Low (20–23K)', 'Mid-High (23–27K)', 'High (>27K)'])
df['Age_Group'] = pd.cut(df['Age'], bins=[17, 30, 40, 50, 65],
                          labels=['18–30', '31–40', '41–50', '51+'])

pivot = df.groupby(['Age_Group', 'Income_Band'], observed=True)['ProdTaken'].mean().unstack() * 100

plt.figure(figsize=(11, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Conversion Rate (%)'})
plt.title('🎯 Conversion Rate Heatmap: Age Group × Income Band\n(higher = better target for sales outreach)',
          fontweight='bold', fontsize=13)
plt.xlabel('Monthly Income Band', fontweight='bold')
plt.ylabel('Age Group', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 HOW TO USE THIS: Sales team should prioritize cells with highest conversion rates.')
print('   These cells represent the best ROI segments for the Wellness Package launch.')

---
## 🏆 Step 8: Business Recommendations

> These are not just data insights — they are **actionable strategies** directly tied to increasing conversion rate and revenue.

In [ ]:
recommendations = [
    {
        'id': 1,
        'title': '🎯 Target Salaried Executives (Age 30–45, Income ₹20K–₹27K) for Basic & Deluxe',
        'evidence': f'Conversion rate for Executives: {df[df["Designation"]=="Executive"]["ProdTaken"].mean()*100:.1f}% vs overall 18%',
        'action': 'Prioritize outreach to this segment. Offer EMI/flexible payment for Basic/Deluxe to remove price friction.'
    },
    {
        'id': 2,
        'title': '📞 Optimal Follow-up: 3–4 Touchpoints (Not More, Not Less)',
        'evidence': f'Conversion peaks at 3-4 follow-ups, then declines — more calls hurt ROI',
        'action': 'Train sales team to close within 3–4 follow-ups. Automate follow-up reminders with decreasing intervals.'
    },
    {
        'id': 3,
        'title': '🌍 Passport Holders are High-Value — Wellness Package Primary Target',
        'evidence': f'Passport holders: {df[df["Passport"]==1]["ProdTaken"].mean()*100:.1f}% conversion vs {df[df["Passport"]==0]["ProdTaken"].mean()*100:.1f}% non-holders',
        'action': 'For Wellness Package launch, build targeted list of passport-holding salaried customers aged 35–55.'
    },
    {
        'id': 4,
        'title': '🏙️ Tier-1 + Tier-3 Cities Drive Highest Volume — Different Strategies Needed',
        'evidence': f'Tier-1: {df[df["CityTier"]==1]["ProdTaken"].mean()*100:.1f}% conv, Tier-3: {df[df["CityTier"]==3]["ProdTaken"].mean()*100:.1f}% conv, Tier-2: lowest',
        'action': 'Tier-1: focus on premium packages (Super Deluxe, King). Tier-3: focus on Basic/Standard with value messaging.'
    },
    {
        'id': 5,
        'title': '👨‍👩‍👧 Family Groups of 3 are the Sweet Spot — Build Family Packages',
        'evidence': 'Group size 3 consistently shows highest purchase rates across all products',
        'action': 'Introduce "Family of 3" bundle pricing for Wellness Package. Highlight kid-friendly wellness activities.'
    },
    {
        'id': 6,
        'title': '💡 Single Customers are Underserved — Create Solo Wellness Tier',
        'evidence': f'Single customers convert at {df[df["MaritalStatus"]=="Single"]["ProdTaken"].mean()*100:.1f}% — above average',
        'action': 'Launch a "Solo Wellness Escape" sub-package at 60-70% of the standard price. Target singles 25–38.'
    }
]

print('='*70)
print('  📋 BUSINESS RECOMMENDATIONS — WELLNESS PACKAGE LAUNCH STRATEGY')
print('='*70)
for rec in recommendations:
    print(f'\n[{rec["id"]}] {rec["title"]}')
    print(f'   EVIDENCE: {rec["evidence"]}')
    print(f'   ACTION:   {rec["action"]}')

In [ ]:
# ── Final Summary Dashboard ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# 1. Conversion by designation
conv1 = df.groupby('Designation')['ProdTaken'].mean().sort_values() * 100
axes[0].barh(conv1.index, conv1.values, color='#3498db', edgecolor='white')
axes[0].axvline(18, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('Conversion by Designation', fontweight='bold')
axes[0].set_xlabel('Conversion Rate (%)')

# 2. Conversion by passport
conv2 = df.groupby('Passport')['ProdTaken'].mean() * 100
axes[1].bar(['No Passport', 'Has Passport'], conv2.values, color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[1].axhline(18, color='gray', linestyle='--', alpha=0.7)
axes[1].set_title('Conversion: Passport vs No Passport', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
for i, val in enumerate(conv2.values):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')

# 3. Conversion by marital status
conv3 = df.groupby('MaritalStatus')['ProdTaken'].mean().sort_values(ascending=False) * 100
axes[2].bar(conv3.index, conv3.values, color=sns.color_palette('pastel'), edgecolor='white')
axes[2].axhline(18, color='red', linestyle='--', alpha=0.7)
axes[2].set_title('Conversion by Marital Status', fontweight='bold')
axes[2].set_ylabel('Conversion Rate (%)')

# 4. Package volume vs conversion
pkg_stats = df.groupby('ProductPitched').agg(
    Total=('ProdTaken', 'count'),
    ConvRate=('ProdTaken', 'mean')
).reset_index()
axes[3].scatter(pkg_stats['Total'], pkg_stats['ConvRate']*100,
                s=200, c=[package_colors[p] for p in pkg_stats['ProductPitched']],
                zorder=5)
for _, row in pkg_stats.iterrows():
    axes[3].annotate(row['ProductPitched'], (row['Total'], row['ConvRate']*100),
                     textcoords='offset points', xytext=(5, 5), fontsize=10)
axes[3].set_title('Package: Volume vs Conversion Rate', fontweight='bold')
axes[3].set_xlabel('Number of Customers Pitched')
axes[3].set_ylabel('Conversion Rate (%)')

# 5. City tier breakdown
ct = df.groupby('CityTier')['ProdTaken'].agg(['count', 'sum'])
ct.columns = ['Total', 'Buyers']
ct['NotBuyers'] = ct['Total'] - ct['Buyers']
ct[['Buyers', 'NotBuyers']].plot(kind='bar', ax=axes[4], color=['#2ecc71', '#e74c3c'],
                                   edgecolor='white', stacked=True)
axes[4].set_title('Buyers vs Non-Buyers by City Tier', fontweight='bold')
axes[4].set_xlabel('City Tier')
axes[4].set_ylabel('Count')
axes[4].tick_params(axis='x', rotation=0)
axes[4].legend(['Bought', 'Did Not Buy'])

# 6. Number of trips vs purchase
conv_trips = df.groupby('NumberOfTrips')['ProdTaken'].mean() * 100
axes[5].plot(conv_trips.index, conv_trips.values, 'o-', color='#9b59b6', linewidth=2, markersize=7)
axes[5].fill_between(conv_trips.index, conv_trips.values, alpha=0.15, color='#9b59b6')
axes[5].set_title('Conversion Rate by Number of Trips/Year', fontweight='bold')
axes[5].set_xlabel('Number of Trips/Year')
axes[5].set_ylabel('Conversion Rate (%)')
axes[5].axhline(18, color='red', linestyle='--', alpha=0.5)

fig.suptitle('📊 Executive Summary Dashboard — Key Conversion Drivers', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✅ Summary of Findings

| Insight | Finding | Business Action |
|---|---|---|
| Overall conversion | 18% baseline | Room to improve with targeted outreach |
| Best demographic | Executives, age 30–45, ₹20K–₹27K | Primary sales target |
| Passport = intent signal | Passport holders convert 2x more | Use passport data as lead qualifier |
| Optimal follow-ups | 3–4 calls is sweet spot | Sales SOP: close within 4 contacts |
| Family groups | Size-3 groups buy most | Bundle pricing for families |
| Wellness target | Salaried, 35–55, Tier-1, passport | Ideal launch segment |
| City focus | Tier-1 for premium, Tier-3 for volume | Differentiated pitch per tier |

---
*Analyst: Rhythm Sharma | IIT Jodhpur | Tools: Python, Pandas, Seaborn, Matplotlib*